In [2]:
import dspy

In [ ]:
lm = dspy.LM(
    'ollama_chat/tiger-gemma2',
    api_base='http://localhost:11434',
)
dspy.configure(lm=lm)

# DSPy modules for various tasks
- https://dspy.ai/#__tabbed_2_4

## Math

In [34]:
math = dspy.ChainOfThought("question -> answer: float")
math(question="Two dice are tossed. What is the probability that the sum equals two?")

Prediction(
    reasoning='There are 36 possible outcomes when rolling two dice. The only way to get a sum of two is by rolling a one on both dice, which occurs once out of the 36 possibilities.',
    answer=0.0278
)

## RAG
- TBD: meta-searchengine?

## Information Extraction

In [35]:
class ExtractInfo(dspy.Signature):
    """Extract structured information from text."""

    text: str = dspy.InputField()
    title: str = dspy.OutputField()
    headings: list[str] = dspy.OutputField()
    entities: list[dict[str, str]] = dspy.OutputField(desc="a list of entities and their metadata")

module = dspy.Predict(ExtractInfo)

text = "Apple Inc. announced its latest iPhone 14 today." \
    "The CEO, Tim Cook, highlighted its new features in a press release."
response = module(text=text)

print(response.title)
print(response.headings)
print(response.entities)

Apple Inc. announces new iPhone 14
['Apple Inc. announced its latest iPhone 14 today.', 'The CEO, Tim Cook, highlighted its new features in a press release.']
[{'name': 'Apple Inc.', 'type': 'company'}, {'name': 'iPhone 14', 'type': 'product'}, {'name': 'Tim Cook', 'type': 'person'}]


## Agents
- TBD: meta-search engine?

## Multi-stage pipeline

In [36]:
class Outline(dspy.Signature):
    """Outline a thorough overview of a topic."""

    topic: str = dspy.InputField()
    title: str = dspy.OutputField()
    sections: list[str] = dspy.OutputField()
    section_subheadings: dict[str, list[str]] = dspy.OutputField(desc="mapping from section headings to subheadings")

class DraftSection(dspy.Signature):
    """Draft a top-level section of an article."""

    topic: str = dspy.InputField()
    section_heading: str = dspy.InputField()
    section_subheadings: list[str] = dspy.InputField()
    content: str = dspy.OutputField(desc="markdown-formatted section")

class DraftArticle(dspy.Module):
    def __init__(self):
        self.build_outline = dspy.ChainOfThought(Outline)
        self.draft_section = dspy.ChainOfThought(DraftSection)

    def forward(self, topic):
        outline = self.build_outline(topic=topic)
        sections = []
        for heading, subheadings in outline.section_subheadings.items():
            section, subheadings = f"## {heading}", [f"### {subheading}" for subheading in subheadings]
            section = self.draft_section(topic=outline.title, section_heading=section, section_subheadings=subheadings)
            sections.append(section.content)
        return dspy.Prediction(title=outline.title, sections=sections)

draft_article = DraftArticle()
article = draft_article(topic="World Cup 2002")

In [37]:
article

Prediction(
    title='World Cup 2002',
    sections=["This section provides a comprehensive overview of the 2002 FIFA World Cup, covering essential details like the host nations, dates, participating teams, and the tournament's format. Let's dive into each aspect:\n\n### Host Nations\n\nThe 2002 FIFA World Cup was jointly hosted by South Korea and Japan, marking the first time that two countries shared hosting duties for a World Cup. This decision aimed to promote football in Asia and showcase the region's growing passion for the sport.\n\n### Dates\n\nThe tournament took place from May 31st to June 30th, 2002, spanning over a month of intense competition. The opening match was held on May 31st at Seoul's World Cup Stadium, while the final match concluded on June 30th at Yokohama International Stadium in Japan.\n\n### Teams\n\nA total of 32 national teams participated in the 2002 FIFA World Cup, representing six continental confederations: Africa (CAF), Asia (AFC), Europe (UEFA), Nort

In [38]:
article.sections

["This section provides a comprehensive overview of the 2002 FIFA World Cup, covering essential details like the host nations, dates, participating teams, and the tournament's format. Let's dive into each aspect:\n\n### Host Nations\n\nThe 2002 FIFA World Cup was jointly hosted by South Korea and Japan, marking the first time that two countries shared hosting duties for a World Cup. This decision aimed to promote football in Asia and showcase the region's growing passion for the sport.\n\n### Dates\n\nThe tournament took place from May 31st to June 30th, 2002, spanning over a month of intense competition. The opening match was held on May 31st at Seoul's World Cup Stadium, while the final match concluded on June 30th at Yokohama International Stadium in Japan.\n\n### Teams\n\nA total of 32 national teams participated in the 2002 FIFA World Cup, representing six continental confederations: Africa (CAF), Asia (AFC), Europe (UEFA), North America and Central America (CONCACAF), South Ameri